# 02. Data Preparation and Preprocessing
**Crop Yield Prediction and Forecasting**  
*Author: Aryan Puranik*

This notebook prepares the historical agricultural dataset for modeling. It implements:
1. **Chronological Splitting** (Train: 1966–2011, Validation: 2012–2014, Test: 2015–2017) to prevent temporal data leakage.
2. **Training-Only Outlier Treatment** to eliminate extreme data-entry errors without contaminating validation/test sets.
3. **Domain Feature Engineering** (climate and nutrient ratios).
4. **Exporting Clean Splits** to `data/processed/`.

> For data audit methodology and outlier investigations, refer to [docs/01_data_audit.md](../docs/01_data_audit.md).

## 1. Setup and Load Raw Data

In [1]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Robust project root discovery
project_root = Path.cwd() if Path.cwd().name != "notebooks" else Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.utils import load_raw_data, split_data_chronological

raw_df = load_raw_data(str(project_root / "data" / "raw" / "crop_yield_dataset.csv"))
print(f"Raw dataset loaded: {raw_df.shape[0]:,} rows, {raw_df.shape[1]} columns")

Raw dataset loaded: 198,369 rows, 15 columns


## 2. Chronological Splitting (Preventing Data Leakage)

In [2]:
# Split chronologically: Train <= 2011, Val 2012-2014, Test 2015-2017
train_df, val_df, test_df = split_data_chronological(raw_df, train_end_year=2011, val_end_year=2014)

total_rows = len(raw_df)
print(f"Train Set:      {len(train_df):,} rows ({len(train_df)/total_rows*100:.1f}%) | Years {train_df['Year'].min()}-{train_df['Year'].max()}")
print(f"Validation Set: {len(val_df):,} rows ({len(val_df)/total_rows*100:.1f}%)  | Years {val_df['Year'].min()}-{val_df['Year'].max()}")
print(f"Test Set:       {len(test_df):,} rows ({len(test_df)/total_rows*100:.1f}%)  | Years {test_df['Year'].min()}-{test_df['Year'].max()}")

Train Set:      176,196 rows (88.8%) | Years 1966-2011
Validation Set: 11,345 rows (5.7%)  | Years 2012-2014
Test Set:       10,828 rows (5.5%)  | Years 2015-2017


**Why Chronological Splitting?**  
In time-series agricultural forecasting, random k-fold cross-validation leaks future technology, irrigation, and weather patterns into the training set. A strict chronological cut mirrors real-world deployment.

## 3. Training-Only Outlier Treatment

In [3]:
# Identify extreme anomalies in the training set
extreme_chickpea = train_df[(train_df["Crop"] == "Chickpea") & (train_df["Yield"] > 5000)]
print("Extreme Chickpea values in training set (median Chickpea yield is ~675 kg/ha):")
display(extreme_chickpea[["Year", "Region", "Crop", "Yield"]])

# Remove these 2 data-entry errors exclusively from the training set
train_clean = train_df.drop(extreme_chickpea.index).copy().reset_index(drop=True)
print(f"\nRemoved {len(extreme_chickpea)} extreme training errors. Clean train shape: {train_clean.shape}")

# Validation and Test sets are NEVER modified to preserve authentic evaluation
val_clean = val_df.copy().reset_index(drop=True)
test_clean = test_df.copy().reset_index(drop=True)

Extreme Chickpea values in training set (median Chickpea yield is ~675 kg/ha):


,Year,Region,Crop,Yield
13679,2007,"Muzaffarpur, Bihar",Chickpea,58363.64
15322,2007,"Saharsa, Bihar",Chickpea,32000.00



Removed 2 extreme training errors. Clean train shape: (176194, 15)


**Important Rule:**  
Outlier treatments must only be applied to training data. Cleaning test data produces an unrealistically clean test set that misrepresents model performance in production.

## 4. Feature Engineering (Domain Indicators)

In [4]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df_out = df.copy()
    # Moisture balance: Rainfall per unit temperature
    df_out["Rainfall_per_Temp"] = (df_out["Rainfall"] / (df_out["Temperature"] + 1e-5)).round(3)
    # Nutrient proportions
    df_out["NP_Ratio"] = (df_out["Nitrogen"] / (df_out["Phosphorus"] + 1e-5)).round(3)
    df_out["NK_Ratio"] = (df_out["Nitrogen"] / (df_out["Potassium"] + 1e-5)).round(3)
    return df_out

train_proc = engineer_features(train_clean)
val_proc = engineer_features(val_clean)
test_proc = engineer_features(test_clean)

print("Engineered features sample:")
train_proc[["Year", "Crop", "Rainfall_per_Temp", "NP_Ratio", "NK_Ratio", "Yield"]].head()

Engineered features sample:


,Year,Crop,Rainfall_per_Temp,NP_Ratio,NK_Ratio,Yield
0,1966,Castor,24.132,10.0,280000.0,192.66
1,1966,Chickpea,24.132,10.0,280000.0,500.00
2,1966,Cotton,24.132,10.0,280000.0,47.62
3,1966,Finger Millet,24.132,10.0,280000.0,743.59
4,1966,Groundnut,24.132,10.0,280000.0,654.76


## 5. Export Processed Datasets

In [5]:
output_dir = project_root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

train_path = output_dir / "train.csv"
val_path = output_dir / "val.csv"
test_path = output_dir / "test.csv"

train_proc.to_csv(train_path, index=False)
val_proc.to_csv(val_path, index=False)
test_proc.to_csv(test_path, index=False)

print(f"Saved: {train_path.name} ({len(train_proc):,} rows, {train_path.stat().st_size / 1e6:.1f} MB)")
print(f"Saved: {val_path.name} ({len(val_proc):,} rows, {val_path.stat().st_size / 1e6:.1f} MB)")
print(f"Saved: {test_path.name} ({len(test_proc):,} rows, {test_path.stat().st_size / 1e6:.1f} MB)")

Saved: train.csv (176,194 rows, 23.3 MB)
Saved: val.csv (11,345 rows, 1.5 MB)
Saved: test.csv (10,828 rows, 1.4 MB)


## 6. Summary & Next Steps

1. **Chronological Partition:** Splitting was completed across 1966–2011 (train), 2012–2014 (val), and 2015–2017 (test).
2. **Leakage Protection:** Extreme data-entry errors were addressed on the training split only.
3. **Engineered Features:** Moisture index (`Rainfall_per_Temp`) and nutrient ratios (`NP_Ratio`, `NK_Ratio`) were generated.
4. **Processed Files:** Clean splits are saved in `data/processed/`.

**Next Notebook:** [03_feature_selection.ipynb](03_feature_selection.ipynb) evaluates feature importance and tests collinearity reduction.